# Feature Engineering & Preprocessing Pipeline
### Customer Churn — Scikit-learn implementation

**Goal:** build a reusable, leakage-aware preprocessing pipeline that handles missing values, scales numeric columns, one-hot encodes categorical columns, examines numeric correlations, and ranks feature importance.

**Important limitation:** the supplied dataset contains only 12 records. This notebook is an educational implementation; any model scores and feature rankings are unstable and must not be used for real customer decisions.

## 1. Imports and configuration

Run this notebook in Jupyter Notebook, JupyterLab, or Google Colab. Keep `customer-churn-training.csv` in the same folder as this notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score
)

RANDOM_STATE = 42
DATA_PATH = "customer-churn-training.csv"
TARGET = "churned"
IDENTIFIER = "customer_id"

## 2. Load the dataset and inspect its structure

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())
display(df.dtypes.to_frame("data_type"))

## 3. Data-quality checks

In [ ]:
print("Missing values by column:")
display(df.isna().sum().to_frame("missing_count"))

print("Duplicate rows:", int(df.duplicated().sum()))
if IDENTIFIER in df.columns:
    print("Unique customer IDs:", df[IDENTIFIER].nunique())

print("\\nTarget distribution:")
display(df[TARGET].value_counts(dropna=False).sort_index().rename(
    index={0: "No churn (0)", 1: "Churn (1)"}
).to_frame("count"))

display(df.describe(include="all").T)

## 4. Separate target and split before fitting transformations

`customer_id` is an identifier, not a predictive feature, so it is removed. The train/test split happens **before** imputers, scaling, or encoding are fitted. Those transformations are kept inside a Scikit-learn pipeline so they learn from training data only.

The stratified split is suitable only for demonstrating code here. With 12 rows, the test set is too small for reliable evaluation.

In [ ]:
if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' is missing.")
if IDENTIFIER in df.columns:
    X = df.drop(columns=[TARGET, IDENTIFIER])
else:
    X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Numeric columns:", X_train.select_dtypes(include=np.number).columns.tolist())
print("Categorical columns:", X_train.select_dtypes(exclude=np.number).columns.tolist())

## 5. Build a reusable preprocessing pipeline

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# A complete pipeline: preprocessing is fitted only on X_train.
model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced",
                                      random_state=RANDOM_STATE))
])

model.fit(X_train, y_train)
print("Preprocessing pipeline fitted successfully.")

## 6. Check transformed feature names and shape

In [ ]:
fitted_preprocessor = model.named_steps["preprocessing"]
transformed_train = fitted_preprocessor.transform(X_train)

try:
    transformed_feature_names = fitted_preprocessor.get_feature_names_out()
except AttributeError:
    transformed_feature_names = [f"feature_{i}" for i in range(transformed_train.shape[1])]

print("Transformed training shape:", transformed_train.shape)
display(pd.DataFrame({
    "transformed_feature": transformed_feature_names
}))

## 7. Evaluate the model carefully

The metrics below are included to demonstrate standard evaluation. Because the holdout contains only a few records, one changed prediction can dramatically change the results. Do not interpret these as evidence of real-world predictive performance.

In [ ]:
predictions = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, predictions), 3))
print("Precision:", round(precision_score(y_test, predictions, zero_division=0), 3))
print("Recall:", round(recall_score(y_test, predictions, zero_division=0), 3))
print("F1 score:", round(f1_score(y_test, predictions, zero_division=0), 3))
print("\\nConfusion matrix (rows=actual, columns=predicted; labels 0, 1):")
print(confusion_matrix(y_test, predictions, labels=[0, 1]))
print("\\nClassification report:")
print(classification_report(
    y_test, predictions, labels=[0, 1],
    target_names=["No churn", "Churn"], zero_division=0
))
print("CAUTION: these metrics are unstable because the dataset is extremely small.")

## 8. Correlation analysis

In [ ]:
# Correlation is computed for numeric columns only; the target is shown for context.
numeric_for_corr = df.select_dtypes(include=np.number)
corr = numeric_for_corr.corr(numeric_only=True)

display(corr.round(2))

fig, ax = plt.subplots(figsize=(7, 5))
image = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(corr.index)
ax.set_title("Numeric Feature Correlation Matrix")
fig.colorbar(image, ax=ax, label="Correlation")
plt.tight_layout()
plt.show()

print("Correlation indicates association, not causation. With 12 rows, patterns can be misleading.")

## 9. Tree-based feature importance

For a demonstration, fit a Random Forest using the **same train-only preprocessing pattern**. Feature importance is reported for the transformed columns (including one-hot encoded plan categories). Rankings from 12 records are not reliable evidence of causal influence or general importance.

In [ ]:
importance_model = Pipeline(steps=[
    ("preprocessing", ColumnTransformer(transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ])),
    ("classifier", RandomForestClassifier(
        n_estimators=200, random_state=RANDOM_STATE, class_weight="balanced"
    ))
])

importance_model.fit(X_train, y_train)
rf_preprocessor = importance_model.named_steps["preprocessing"]
rf_classifier = importance_model.named_steps["classifier"]

try:
    feature_names = rf_preprocessor.get_feature_names_out()
except AttributeError:
    feature_names = [f"feature_{i}" for i in range(len(rf_classifier.feature_importances_))]

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_classifier.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)

display(importance_df)

ax = importance_df.sort_values("importance").plot(
    kind="barh", x="feature", y="importance", legend=False, figsize=(8, 5)
)
ax.set_title("Random Forest Feature Importance (Educational Only)")
ax.set_xlabel("Importance")
ax.set_ylabel("Transformed feature")
plt.tight_layout()
plt.show()

## 10. Reuse the fitted pipeline on new records

In [ ]:
# Example only. Replace values with valid, authorized data using the same input columns.
# Do not include customer_id or the target column.
example_customer = pd.DataFrame([{
    "tenure_months": 5,
    "support_tickets": 3,
    "monthly_spend_inr": 799,
    "last_login_days": 15,
    "plan_type": "Basic"
}])

# Ensure column order matches the training input.
example_customer = example_customer[X_train.columns]
predicted_class = int(model.predict(example_customer)[0])
predicted_probability = float(model.predict_proba(example_customer)[0, 1])

print("Predicted class (1=churn, 0=no churn):", predicted_class)
print("Illustrative churn probability:", round(predicted_probability, 3))
print("This output is not validated for operational use; any action requires human review.")

## 11. Responsible-use checklist

- [x] Remove `customer_id` from model features.
- [x] Split training and test data before fitting transformations.
- [x] Impute missing numeric values with the training median.
- [x] Scale numeric features with `StandardScaler`.
- [x] Impute missing categorical values with the training most-frequent category.
- [x] One-hot encode categories and ignore unseen categories safely.
- [x] Review numeric correlations and tree-based feature importance.
- [ ] Verify data source, licence, consent/legal basis, and permitted use with the provider.
- [ ] Verify that each feature was recorded before the churn prediction point; remove post-outcome information.
- [ ] Evaluate on a larger, representative, independent dataset before any real-world use.
- [ ] Conduct an appropriate fairness review when lawful, relevant data and sufficient sample size are available.

**Conclusion:** this notebook implements the requested preprocessing and feature-engineering workflow. The 12-row dataset is only suitable for learning and code demonstration, not for business decisions.